# Model Serving — Deploy e Invocação do Endpoint

Este notebook cria um **endpoint de Model Serving** no Databricks para o modelo de risco de crédito treinado e registrado no Unity Catalog (notebook `3-Treinamento`).

**Etapas:**
1. Configuração das variáveis (catálogo, schema, prefixo)
2. Instalação de dependências
3. Criação do endpoint de serving
4. Verificação do status do endpoint
5. Invocação do endpoint com dados de exemplo

In [0]:
%pip install databricks-sdk==0.36.0 mlflow==2.19.0
dbutils.library.restartPython()

In [0]:
# ============================================================
# Preencha apenas estas variáveis (mesmo padrão do 3-Treinamento)
# ============================================================
CATALOGO = ""
SCHEMA = ""
PREFIXO = ""


# Nomes derivados
model_name = f"{CATALOGO}.{SCHEMA}.{PREFIXO}_credit_classification"
endpoint_name = f"{PREFIXO}-credit-risk-serving"

## Criação do Endpoint de Model Serving

Utilizamos o **Databricks SDK** para criar um endpoint que serve o modelo registrado no Unity Catalog.

Parâmetros principais:
- **`entity_name`**: nome completo do modelo no Unity Catalog (`catalogo.schema.modelo`)
- **`entity_version`**: versão do modelo a ser servida (usamos a última versão com alias `prod`)
- **`workload_size`**: tamanho da infra (`Small`, `Medium`, `Large`)
- **`scale_to_zero_enabled`**: permite reduzir custo quando não há requisições

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)
import mlflow
from mlflow import MlflowClient


# Descobrir a versão do modelo com alias "prod"
mlflow.set_registry_uri("databricks-uc")
client_mlflow = MlflowClient()
model_version = client_mlflow.get_model_version_by_alias(model_name, "prod").version
print(f"Modelo: {model_name} | Versão (prod): {model_version}")

# Criar o endpoint de serving
w = WorkspaceClient()

try:
    print(f"\nCriando endpoint '{endpoint_name}'...")
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=[
                ServedEntityInput(
                    name=f"{PREFIXO}-credit-entity",
                    entity_name=model_name,
                    entity_version=model_version,
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                )
            ]
        ),
    )
    print(f"Endpoint '{endpoint_name}' criado e pronto!")
except Exception as e:
    if "already exists" in str(e).lower() or "RESOURCE_ALREADY_EXISTS" in str(e):
        print(f"Endpoint '{endpoint_name}' já existe. Atualizando configuração...")
        w.serving_endpoints.update_config_and_wait(
            name=endpoint_name,
            served_entities=[
                ServedEntityInput(
                    name=f"{PREFIXO}-credit-entity",
                    entity_name=model_name,
                    entity_version=model_version,
                    workload_size="Small",
                    scale_to_zero_enabled=True,
                )
            ],
        )
        print(f"Endpoint '{endpoint_name}' atualizado com sucesso!")
    else:
        raise e

## Verificar status do endpoint

Após a criação, confirmamos que o endpoint está no estado **READY** antes de realizar invocações.

In [0]:
# Verificar o status do endpoint
endpoint_info = w.serving_endpoints.get(name=endpoint_name)
print(f"Endpoint: {endpoint_info.name}")
print(f"Estado:   {endpoint_info.state.ready}")
print(f"URL:      {endpoint_info.url if hasattr(endpoint_info, 'url') else 'N/A'}")

# Detalhes das entidades servidas
for entity in endpoint_info.config.served_entities:
    print(f"\n  Entidade: {entity.name}")
    print(f"  Modelo:   {entity.entity_name} v{entity.entity_version}")
    print(f"  Tamanho:  {entity.workload_size}")
    print(f"  Scale-to-zero: {entity.scale_to_zero_enabled}")

## Invocação do Endpoint

Agora vamos invocar o endpoint com dados de exemplo para obter predições em tempo real.

Usamos o método `predict()` do **MLflow Deployments SDK**, que aceita dados no formato `dataframe_split` (colunas + dados).

In [0]:
import mlflow.deployments
import pandas as pd

# Carregar dados de exemplo da tabela de treino para construir o payload
df_exemplo = spark.table(f"{CATALOGO}.{SCHEMA}.{PREFIXO}credit_risk_train_df").drop("defaulted").limit(5).toPandas()

print("Dados enviados ao endpoint:")
print(df_exemplo.to_string(index=False))
print(f"\nColunas: {list(df_exemplo.columns)}")
print(f"Registros: {len(df_exemplo)}")

In [0]:
# Invocar o endpoint de serving
deploy_client = mlflow.deployments.get_deploy_client("databricks")

response = deploy_client.predict(
    endpoint=endpoint_name,
    inputs={"dataframe_split": df_exemplo.to_dict(orient="split")},
)

# Exibir resultados
print("Resposta do endpoint (predições):")
print(response)

# Montar DataFrame com resultados
resultado = pd.DataFrame({
    "predicao_risco": response["predictions"] if isinstance(response, dict) else response
})
resultado = pd.concat([df_exemplo.reset_index(drop=True), resultado], axis=1)
display(spark.createDataFrame(resultado))

## Próximos passos

- **Monitoramento**: ative as *inference tables* para capturar requests/responses automaticamente
- **Autoscaling**: ajuste `workload_size` e concorrência conforme a demanda
- **Integração**: use a URL do endpoint em aplicações externas (APIs, apps Streamlit, etc.)
- **Atualização**: ao retreinar o modelo, basta registrar nova versão com alias `prod` e atualizar o endpoint